In [49]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [50]:
#Extract pdfs
def load_pdf(path):
    loader = DirectoryLoader(
        path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents


In [51]:
loaded_data = load_pdf("../data")
loaded_data

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 7 0 (offset 0)


[Document(metadata={'producer': '3-Heights(TM) PDF Security Shell 4.8.25.2 (http://www.pdf-tools.com) / pdcat (www.pdf-tools.com)', 'creator': 'Pages', 'creationdate': "D:20240315121206Z00'00'", 'title': 'Acts of Parliament 2022', 'moddate': '2024-03-15T12:31:58+00:00', 'source': '..\\data\\Acts-of-Parliament-2022.pdf', 'total_pages': 614, 'page': 0, 'page_label': '1'}, page_content='ACTS OF PARLIAMENT \n2022\nCompiled by LankaLAW www.lankalaw.net'),
 Document(metadata={'producer': '3-Heights(TM) PDF Security Shell 4.8.25.2 (http://www.pdf-tools.com) / pdcat (www.pdf-tools.com)', 'creator': 'Pages', 'creationdate': "D:20240315121206Z00'00'", 'title': 'Acts of Parliament 2022', 'moddate': '2024-03-15T12:31:58+00:00', 'source': '..\\data\\Acts-of-Parliament-2022.pdf', 'total_pages': 614, 'page': 1, 'page_label': '2'}, page_content='List of Acts\n01/2022 : Mahapola Higher Education Scholarship Trust \nFund (Amendment)\n02/2022 : Code of Criminal Procedure (Amendment)\n03/2022 : PROHIBITIO

In [52]:
len(loaded_data)

2438

In [53]:
# filter content(improverd filter)
from typing import List
from langchain.schema import Document
import re

NOISE_PATTERNS = [
    r"^list of acts",
    r"^table of contents",
    r"printed by the department",
    r"published by.*government",
    r"extraordinary gazette",
    r"price\s*:\s*rs",
    r"certified on.*\d{4}",
    r"^\s*page\s+\d+\s*$",
    r"©.*all rights reserved",
    r"can be purchased at",
    r"prakashana piyasa",
    r"compiled by lankalaw",
]

LEGAL_PATTERNS = [
    r"\bact no\.?\s*\d+",
    r"\bsection\s+\d+",
    r"\bsubsection\s*\(\d+\)",
    r"\bclause\s*\([a-z]\)",
    r"\bparagraph\s*\([ivx]+\)",
    r"\bchapter\s+[IVX]+",
    r"\bpart\s+[IVX]+",
    r"\benact(ed|s|ment)\b",
    r"\bamend(ed|ment|s)\b",
    r"\brepealed?\b",
    r"\bprovided that\b",
    r"\bnotwithstanding\b",
    r"\bwhereas\b",
    r"\bhereby\b",
    r"\bshall\b.*\b(be|have|not)\b",
    r"\bmay\b.*\b(be|make|appoint)\b",
    r"\bmeans\b.*includes?",
    r"\boffence\b.*\bpunishable\b",
    r"\bpenalty\b.*fine",
    r"\bimprisonment\b",
]

WEAK_LEGAL_SIGNALS = [
    r"\bshall\b", r"\bmay\b", r"\bmust\b", r"\bperson\b",
    r"\bauthority\b", r"\bminister\b", r"\bcourt\b", r"\btribunal\b",
]

def doc_filter(docs: List[Document]) -> List[Document]:
    filtered_docs = []
    print(f"Filtering {len(docs)} documents...")

    for doc in docs:
        text = doc.page_content.strip()
        text_lower = text.lower()

        if len(text) < 100:
            continue

        noise_matches = sum(1 for p in NOISE_PATTERNS if re.search(p, text_lower))
        if noise_matches >= 1:  # STRICTER
            continue

        strong_legal = sum(1 for p in LEGAL_PATTERNS if re.search(p, text_lower))
        weak_legal = sum(1 for p in WEAK_LEGAL_SIGNALS if re.search(p, text_lower))

        if strong_legal >= 1 or weak_legal >= 4:  # RAISED threshold
            filtered_docs.append(doc)

    print(f"Kept {len(filtered_docs)}/{len(docs)} ({len(filtered_docs)/len(docs)*100:.1f}%)")
    return filtered_docs


 

In [54]:
filtered_data =doc_filter(loaded_data)
filtered_data

Filtering 2438 documents...
Kept 2171/2438 (89.0%)


[Document(metadata={'producer': '3-Heights(TM) PDF Security Shell 4.8.25.2 (http://www.pdf-tools.com) / pdcat (www.pdf-tools.com)', 'creator': 'Pages', 'creationdate': "D:20240315121206Z00'00'", 'title': 'Acts of Parliament 2022', 'moddate': '2024-03-15T12:31:58+00:00', 'source': '..\\data\\Acts-of-Parliament-2022.pdf', 'total_pages': 614, 'page': 6, 'page_label': '7'}, page_content='2    Mahapola Higher Education Scholarship Trust Fund\n(Amendment) Act, No. 1 of 2022\nother document shall be read and construed as a reference to\nthe “Lalith Athulathmudali Mahapola Higher Education\nScholarship Trust Fund Act” and “Lalith Athulathmudali\nMahapola Higher Education Scholarship Trust Fund”.\n4. Section 3 of the principal enactment is hereby amended\nas follows:-\n(1) by the repeal of subsection (2) thereof and the\nsubstitution therefor of the following subsection:-\n“(2) The Board of Trustees of the Fund (hereinafter\nreferred to as the “Board”) shall consist of the\nfollowing members:-\

In [87]:
# split to chunks

def text_split(doc_list):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 800,
        chunk_overlap=100,
        length_function=len,
        is_separator_regex=False,
    )
    splitted_text = text_splitter.split_documents(doc_list)
    return splitted_text

In [88]:
data_chunks = text_split(filtered_data)
data_chunks

[Document(metadata={'producer': '3-Heights(TM) PDF Security Shell 4.8.25.2 (http://www.pdf-tools.com) / pdcat (www.pdf-tools.com)', 'creator': 'Pages', 'creationdate': "D:20240315121206Z00'00'", 'title': 'Acts of Parliament 2022', 'moddate': '2024-03-15T12:31:58+00:00', 'source': '..\\data\\Acts-of-Parliament-2022.pdf', 'total_pages': 614, 'page': 6, 'page_label': '7'}, page_content='2    Mahapola Higher Education Scholarship Trust Fund\n(Amendment) Act, No. 1 of 2022\nother document shall be read and construed as a reference to\nthe “Lalith Athulathmudali Mahapola Higher Education\nScholarship Trust Fund Act” and “Lalith Athulathmudali\nMahapola Higher Education Scholarship Trust Fund”.\n4. Section 3 of the principal enactment is hereby amended\nas follows:-\n(1) by the repeal of subsection (2) thereof and the\nsubstitution therefor of the following subsection:-\n“(2) The Board of Trustees of the Fund (hereinafter\nreferred to as the “Board”) shall consist of the\nfollowing members:-\

In [89]:
print(f'total chunks: {len(data_chunks)}')

total chunks: 7105


In [ ]:
# embedding model download for data ingestion

from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model= "sentence-transformers/all-MiniLM-L6-v2"
    embeddings=HuggingFaceEmbeddings(model_name= model)

    return embeddings

In [91]:
Embedding = download_embeddings()

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 7347f33d-0cdb-4b95-908c-a30bea9a60ef)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].


In [92]:
# test embedding work properly
vector = Embedding.embed_query("LLM OPs")
vector

[-0.05659736320376396,
 -0.016393793746829033,
 0.009783041663467884,
 0.012276718392968178,
 -0.00938601978123188,
 -0.06348241120576859,
 0.023746583610773087,
 0.008308597840368748,
 -0.03279753401875496,
 0.014701695181429386,
 -0.01219838485121727,
 0.032230354845523834,
 0.027652084827423096,
 -0.012582303956151009,
 0.025902509689331055,
 0.09583776444196701,
 0.09715840220451355,
 0.04905128479003906,
 -0.012099115177989006,
 -0.004134247545152903,
 -0.02012760005891323,
 0.009564577601850033,
 0.04927167668938637,
 -0.010034754872322083,
 -0.01192828081548214,
 -0.01188508328050375,
 0.017794793471693993,
 0.11908303201198578,
 -0.026031140238046646,
 -0.06363314390182495,
 0.04246900603175163,
 0.035428907722234726,
 0.10718737542629242,
 0.02880008891224861,
 -0.0038097361102700233,
 -0.026931248605251312,
 -0.04276687651872635,
 -0.011387919075787067,
 -0.014532582834362984,
 -0.017599554732441902,
 0.003245071740821004,
 -0.04637632891535759,
 -0.0030160858295857906,
 -0.0

In [93]:
len(vector)

384

In [94]:
# import api keys

import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("PINECONE_API_KEY"):
    raise RuntimeError("PINECONE_API_KEY NotFound!")

if not os.getenv("LANGCHAIN_API_KEY"):
    raise RuntimeError("LANGCHAIN_API_KEY NotFound!")

if os.getenv("LANGCHAIN_TRACING_V2") != "true":
    raise RuntimeError("LANGCHAIN_TRACING_V2 not set to 'true'!")

In [83]:
# Auth vector db

from pinecone import Pinecone

pinecone_api_key = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)
pc

In [95]:
# db indexing
from pinecone import ServerlessSpec

index_name = "law-rag-genai-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [96]:
# store chunks
from langchain_pinecone import PineconeVectorStore

doc = PineconeVectorStore.from_documents(
    documents=data_chunks,
    embedding=Embedding,
    index_name=index_name
)

In [97]:
# load data from vectorstore

doc_search= PineconeVectorStore.from_existing_index(
    index_name = index_name,
    embedding = Embedding
)

In [100]:
# retriver initilaize

Retriever = doc_search.as_retriever(
    search_type ="similarity",
    search_kwargs={'k':5}
)


In [119]:
# check retriver
retrived_data = Retriever.invoke("punishments of intellectual property law in sri lanka")
retrived_data

[Document(id='eee2ff70-3f60-4595-b98d-3fb045dc3d08', metadata={'creationdate': "D:20250310130835Z00'00'", 'creator': 'Pages', 'moddate': "D:20250310130835Z00'00'", 'page': 0.0, 'page_label': '1', 'producer': 'macOS Version 13.4.1 (c) (Build 22F770820d) Quartz PDFContext', 'source': '..\\data\\Intellectual-Property-Consolidated-2024.pdf', 'title': 'Intellectual Property - Consolidated (2024)', 'total_pages': 144.0}, page_content='INTELLECTUAL PROPERTY ACT \nAN ACT TO PROVIDE FOR THE LAW RELATING TO INTELLECTUAL PROPERTY AND \nFOR AN EFFICIENT PROCEDURE FOR THE REGISTRATION, CONTROL AND \nADMINISTRATION THERE OF ; TO AMEND THE CUSTOMS ORDINANCE (CHAPTER \n235) AND THE HIGH COURT OF THE PROVINCES (SPECIAL) PROVISIONS ACT, NO. \n10 OF 1996; AND TO PROVIDE FOR MATTERS CONNECTED THEREWITH OR \nINCIDENTAL THERETO \nBE it enacted by the Parliament of the Democratic Socialist Republic of Sri Lanka as follows- \nAct Nos, \n7 of 2018 \n8 of 2021 \n8 of 2022 \nShort title. 1. This Act may be cited

In [102]:
# Model initialize
from langchain_ollama import ChatOllama

LLM = ChatOllama(
    model="gemma3:1b",
    temperature=0.1
)

In [103]:
# chains
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate


In [115]:
# prompt template

system_prompt_text =("""
You are a Legal Information Assistant for Sri Lankan law.

RULES:
1. Answer using the retrieved documents.
2. If not found, say: "Information not available in provided documents."
3. Cite sources: mention act name, section, or page number from the context.
4. Use formal legal language.
5. Do NOT use external knowledge.
Context:
{context}
"""
)

system_prompt = SystemMessagePromptTemplate.from_template(system_prompt_text)
human_prompt = HumanMessagePromptTemplate.from_template("{input}")

prompt = ChatPromptTemplate.from_messages(
    [
        system_prompt,
        human_prompt
    ]
)

In [116]:
# chain
QnA_chain = create_stuff_documents_chain(LLM, prompt)
RAG_chain = create_retrieval_chain(Retriever, QnA_chain)


In [118]:
# get responses

question = "what punishments of intellectual property law in sri lanka?"
response = RAG_chain.invoke({"input":question})
print(response['answer'])

Okay, let's delve into the punishments related to intellectual property (IP) law in Sri Lanka, as outlined in the provided text.

Based on the text, the punishments for non-compliance with the Intellectual Property Act are quite stringent, primarily focusing on financial penalties. Here’s a breakdown of the punishments:

*   **Fine:** A company applying to the Act shall be liable to a fine not exceeding fifty thousand rupees. This is a significant penalty, reflecting the seriousness of the law.
*   **Criminal Offense:**  Failure to comply with the provisions of the Act, particularly Section 493, can result in a criminal offense.
*   **Penalty for Willful Breach:**  If the offense is willful and knowing, the offender (officer or agent) shall be liable to a fine of not exceeding one thousand rupees.
*   **Bribes and Bribery:**  The text explicitly states that any officer or person who gives or offers a bribe, gratuity, recompense, or reward for the neglect or non-performance of his duty 

In [120]:
question = "what is LLM?"
response = RAG_chain.invoke({"input":question})
print(response['answer'])

Information not available in provided documents.
